In [1]:
import os
import shutil
import pandas as pd
import re
from lxml import etree

year = 2026
sentido = "Recibidos"   # Puedes fijar "Emitidos" o "Recibidos" directamente
#sentido = "Recibidos"   # Puedes fijar "Emitidos" o "Recibidos" directamente

def clean_xml(file_path):
    """Elimina xmlns:schemaLocation del XML antes de procesarlo."""
    with open(file_path, "r", encoding="utf-8") as f:
        xml_content = f.read()
    return re.sub(r'xmlns:schemaLocation\s*=\s*".*?"', '', xml_content)

def process_and_classify_xml_files(base_directory, d_path, year=year):
    file_list = []
    
    if not os.path.exists(base_directory):
        print(f"🚫 El directorio {base_directory} no existe.")
        return pd.DataFrame()

    # Ya no recorremos meses, solo trabajamos sobre el directorio base
    for root, _, files in os.walk(base_directory):
        for file in files:
            if not file.endswith('.xml'):
                continue

            file_path = os.path.join(root, file)

            try:
                # Limpieza y parseo del XML con recuperación de errores
                cleaned_xml = clean_xml(file_path)
                parser = etree.XMLParser(recover=True)
                root_element = etree.fromstring(cleaned_xml.encode(), parser)

                # Definir namespaces y extraer atributos
                namespaces = {'cfdi': 'http://www.sat.gob.mx/cfd/4', 'tfd': 'http://www.sat.gob.mx/TimbreFiscalDigital'}
                tipo_comprobante = root_element.get('TipoDeComprobante', 'N/A')

                # Extraer UUID
                timbre = root_element.find(".//tfd:TimbreFiscalDigital", namespaces)
                uuid = timbre.attrib.get('UUID', 'N/A') if timbre is not None else 'N/A'

                # Determinar la carpeta de destino
                destination_folder = {
                    'I': 'Trabajo Ingresos',
                    'N': 'Trabajo Nomina',
                    'E': 'Trabajo NCredito',
                    'P': 'Trabajo Pago'
                }.get(tipo_comprobante, 'Trabajo Otros')

                destination_path = os.path.join(d_path, destination_folder)
                os.makedirs(destination_path, exist_ok=True)

                # Generar nuevo nombre de archivo si hay UUID
                new_file_name = f"{uuid}.xml" if uuid != 'N/A' else file
                destination_file_path = os.path.join(destination_path, new_file_name)

                # Copiar el archivo con mejor manejo de permisos
                if os.access(destination_path, os.W_OK):
                    shutil.move(file_path, destination_file_path)
#                    shutil.copy(file_path, destination_file_path)
                else:
                    print(f"🚫 Permiso denegado para escribir en {destination_path}")

                # Agregar datos al DataFrame
                file_list.append({
                    "File Path": file_path,
                    "Path": destination_path,
                    "File Name": new_file_name,
                    "TipoDeComprobante": tipo_comprobante,
                    "UUID": uuid
                })

            except etree.XMLSyntaxError as e:
                print(f"⚠️ Error de sintaxis en XML ({file}): {e}")
            except FileNotFoundError:
                print(f"❌ Archivo no encontrado: {file_path}")
            except PermissionError:
                print(f"🚫 Permiso denegado al acceder a {file_path}")
            except Exception as e:
                print(f"❌ Error desconocido en {file}: {e}")

    # Convertir a DataFrame y guardar en CSV
    df = pd.DataFrame(file_list)
    df = df.sort_values('TipoDeComprobante')
    output_csv_path = os.path.join(d_path, fr'TipoXML {sentido} {year}.csv')
    df.to_csv(output_csv_path, index=False)
    
    print(f"✅ El archivo se ha guardado en: {output_csv_path}")
    return df


# Carpeta Origen
base_directory = fr'C:\Users\RGARCIA\Downloads\SAT 2020\Operaciones\2026 XML\ZIP\260403\Recibidos'
# Carpeta Destino
d_path = fr'C:\Users\RGARCIA\Downloads\SAT 2020\Operaciones\2026 XML\Recibidos'

# Ejecutar la función
df_files = process_and_classify_xml_files(base_directory, d_path, year)


✅ El archivo se ha guardado en: C:\Users\RGARCIA\Downloads\SAT 2020\Operaciones\2026 XML\Recibidos\TipoXML Recibidos 2026.csv
